# DreamBrush 3DGS Training

This notebook turns a DreamBrush `CaptureBundle` into a Nerfstudio-ready dataset, trains a 3D Gaussian Splat (Splatfacto), and exports a PLY for the iOS viewer.

**Pipeline**
1. Download bundle from Google Drive
2. Convert to `transforms.json` + `images/`
3. Build `sparse_pc.ply` from LiDAR depth for better Splatfacto initialization
4. Train (`ns-train splatfacto`)
5. Export PLY (`ns-export gaussian-splat`)

> Note: The app exports camera transforms in **row-major** layout (as required by Nerfstudio), so this notebook does not transpose matrices.

## 0) Environment setup (GPU VM)

This workflow expects a CUDA-capable GPU for training. Run this cell once per VM:

In [ ]:
import os, sys, subprocess, venv, textwrap, importlib

# --- config ---
VENV_DIR = ".venv"
REQS = ["nerfstudio==1.1.5", "gdown", "imageio", "pillow", "tqdm", "matplotlib", "open3d", "pyyaml"]
# --------------

def run(cmd, **kwargs):
    print(">>", " ".join(cmd))
    subprocess.run(cmd, check=True, **kwargs)

# 1) Create venv that can SEE the preinstalled system packages (incl. torch/cuda)
if not os.path.isdir(VENV_DIR):
    print(f"Creating venv at {VENV_DIR} (with system-site-packages)...")
    venv.EnvBuilder(with_pip=True, system_site_packages=True).create(VENV_DIR)
else:
    print(f"Venv already exists at {VENV_DIR}")

vpy = os.path.join(VENV_DIR, "bin", "python")
if not os.path.exists(vpy):
    raise RuntimeError(f"Expected venv python at {vpy}, but it wasn't found.")

# 2) Tooling
run([vpy, "-m", "pip", "install", "-U", "pip", "setuptools", "wheel"])

# 3) Fix the common Ubuntu/apt "distutils blinker 1.4 can't uninstall" issue:
#    shadow it inside the venv instead of trying to remove the system one.
run([vpy, "-m", "pip", "install", "--ignore-installed", "blinker>=1.6"])

# 4) Pin the *system* torch stack so pip won't download/replace it
constraints_path = "/tmp/torch-constraints.txt"
pin_code = r"""
import importlib.metadata as md
pins=[]
for name in ("torch","torchvision","torchaudio"):
    try:
        pins.append(f"{name}=={md.version(name)}")
    except md.PackageNotFoundError:
        pass
path = """ + repr(constraints_path) + r"""
with open(path,"w") as f:
    f.write("\n".join(pins) + ("\n" if pins else ""))
print(path)
print(open(path).read() if pins else "(no torch packages found to pin)")
"""
out = subprocess.check_output([vpy, "-c", pin_code], text=True)
print("Torch constraints:\n" + out)

# 5) Install your deps, respecting the torch constraints (so no slow torch install)
run([vpy, "-m", "pip", "install", "-c", constraints_path, *REQS])

# 6) Make this venv usable as a Jupyter kernel (optional but recommended)
run([vpy, "-m", "pip", "install", "-c", constraints_path, "ipykernel"])
run([vpy, "-m", "ipykernel", "install", "--user", "--name", "runpod-nerf", "--display-name", "Python (runpod-nerf)"])

# 7) Also make venv packages importable *in the current kernel* (no restart needed)
#    (This is a pragmatic hack: we add the venv site-packages to sys.path.)
site_pkgs = subprocess.check_output([vpy, "-c", "import site; print(site.getsitepackages()[0])"], text=True).strip()
if site_pkgs not in sys.path:
    sys.path.insert(0, site_pkgs)
importlib.invalidate_caches()

os.environ["VIRTUAL_ENV"] = os.path.abspath(VENV_DIR)
os.environ["PATH"] = os.path.abspath(os.path.join(VENV_DIR, "bin")) + os.pathsep + os.environ.get("PATH", "")

print("\n✅ Done.")
print("Best practice: switch your notebook kernel to 'Python (runpod-nerf)' for a clean env.")
print("But you can also keep going in this kernel now; venv site-packages were added to sys.path.")

In [ ]:
import subprocess
from pathlib import Path

patch_script = Path.cwd() / "patch_nerfstudio.py"
subprocess.run([str(Path(".venv/bin/python")), str(patch_script)], check=True)
print("✅ Nerfstudio patched.")


## 1) Imports + configuration

Fill in your Google Drive link and choose output paths.

In [ ]:
from __future__ import annotations

import json
import math
import os
import re
import shutil
import zipfile
from pathlib import Path

import numpy as np
import imageio.v2 as imageio
from PIL import Image
from tqdm import tqdm

# ==================== USER CONFIG ====================

# ==================== FRAME SELECTION ====================
# USE_KEYFRAMES=False uses ALL captured frames (recommended for quality)
# USE_KEYFRAMES=True uses only ARKit-selected keyframes (faster, less data)
USE_KEYFRAMES = False      # Changed: use all frames for better coverage
FRAME_STRIDE = 1           # Keep every Nth frame (1 = all, 2 = half, etc.)
MAX_FRAMES = None          # Cap number of frames (None = no limit)

# ==================== POINT CLOUD CONFIG ====================
DEPTH_STRIDE = 2           # Downsample depth pixels (lower = more points, better init)
MIN_DEPTH_M = 0.2          # Minimum depth in meters
MAX_DEPTH_M = 5.0          # Maximum depth (reduced for indoor scenes)
VOXEL_SIZE_M = 0.01        # Voxel size for downsampling (1cm = finer detail)
MAX_POINTS_PER_FRAME = 300_000  # Max points per frame before random sampling

# ==================== DEPTH CONFIDENCE CONFIG ====================
USE_CONFIDENCE = True      # Use per-pixel confidence map if present
MIN_CONFIDENCE = 1         # 0=low, 1=medium, 2=high (use 2 for strict)

# ==================== POSE REFINEMENT ====================
CAMERA_OPTIMIZER_DEVICE = "auto"  # auto = cuda when available
USE_TF32 = True                   # faster matmul/conv on Ada GPUs
ENABLE_CAMERA_OPTIMIZER = True
CAMERA_OPTIMIZER_MODE = "SO3xR3"  # common modes: "SO3xR3" (rot+trans) or "SE3"

# ==================== DEPTH SUPERVISION ====================
ENABLE_DEPTH_SUPERVISION = True
DEPTH_LOSS_WEIGHT = 0.1
DEPTH_LOSS_TYPE = "huber"         # "huber" or "l1"
DEPTH_LOSS_HUBER_DELTA = 0.1      # meters
DEPTH_LOSS_RAMP_STEPS = 2000
DEPTH_SUP_MIN_M = MIN_DEPTH_M
DEPTH_SUP_MAX_M = MAX_DEPTH_M
DEPTH_SUP_MIN_CONFIDENCE = MIN_CONFIDENCE

# ==================== DATA MANAGER (MEMORY/PERF) ====================
GPU_CACHE_MAX_FRACTION = 0.6  # cap GPU cache usage to this fraction of VRAM
PREFETCH_TO_GPU = True         # overlap CPU->GPU transfer with rendering
NON_BLOCKING_TRANSFERS = True  # enable async GPU copies from pinned memory
CV2_NUM_THREADS = None         # set to an int to override OpenCV threads
CACHE_IMAGES_DEVICE = "auto"   # auto: gpu if it fits, else cpu
ALLOW_LARGE_GPU_CACHE = False  # allow auto fallback when datasets are large
CACHE_IMAGES_TYPE = "uint8"    # uint8 saves a lot of RAM
MAX_THREAD_WORKERS = None        # None = use all available CPU cores
DEPTH_RESIZE_MODE = "max_edge"  # native/image/max_edge
DEPTH_MAX_EDGE = 480            # only used when DEPTH_RESIZE_MODE="max_edge"
DEPTH_DTYPE = "float16"       # float16 saves RAM; cast to float32 in loss
CONFIDENCE_DTYPE = "uint8"    # confidence is 0/1/2 so uint8 is enough

# ==================== INTRINSICS ====================
USE_PER_FRAME_INTRINSICS = True
INTRINSICS_SPREAD_THRESHOLD = 0.5  # if fx/fy/cx/cy vary more than this, use per-frame

# ==================== COORDINATE CONVERSION ====================
# If you see mirrored results, try enabling this camera-axis conversion.
APPLY_CAMERA_AXIS_CONVERSION = False
CAMERA_AXIS_CONVERSION = np.diag([1.0, 1.0, -1.0, 1.0]).astype(np.float32)


# ==================== LOGGING ====================
LOGGING_STEPS_PER_LOG = 20

# ==================== TRAINING HYPERPARAMETERS ====================
# These are passed to ns-train splatfacto
TRAINING_CONFIG = {
    # Iteration control
    "max_num_iterations": 30000,      # Default is 30000, increase for better quality
    
    # Culling thresholds - CRITICAL for quality
    # Lower cull_alpha_thresh = keep more semi-transparent gaussians = better quality
    "cull_alpha_thresh": 0.005,       # Default 0.1, lower keeps more splats
    "continue_cull_post_densification": False,  # Don't cull after 15k steps
    
    # Densification - controls how aggressively new gaussians are created
    "densify_grad_thresh": 0.0004,    # Default 0.0008, lower = more aggressive densification
    
    # Resolution schedule - training resolution ramp-up
    "num_downscales": 2,              # Start at 1/4 resolution, ramp up
    "resolution_schedule": 3000,      # Double resolution every N steps
    
    # Refinement schedule
    "refine_every": 100,              # Densify/cull every N steps
    "stop_split_at": 15000,           # Stop splitting gaussians at this step
    
    # Quality vs size tradeoff
    # Note: More gaussians = better quality but larger file & slower mobile rendering
    # iPhone 13 Pro can handle ~1-2M splats at 15fps in Quality mode
    "use_scale_regularization": True, # Reduces spikey artifacts
    "max_gauss_ratio": 5.0,           # Default 10.0, lower = more uniform splats
    
    # Loss weights
    "ssim_lambda": 0.2,               # SSIM loss weight (structural similarity)
}

# ==================== iOS RENDER TARGETS ====================
# Reference for splat count targets:
# - iPhone 13 Pro Quality mode: renders all splats at 15fps
# - iPhone 13 Pro Balanced mode: caps at ~60,000 splats
# - iPhone 13 Pro Fast mode: caps at ~20,000 splats
# Aim for 500K-1.5M total splats for good quality with reasonable mobile performance

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
DATASET_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# ==================== PRECOMPUTE DATASET ====================
USE_PRECOMPUTED_DATASET = False
PRECOMPUTED_DATASET_DIR = WORK_DIR / "nerfstudio_dataset_preprocessed"
PRECOMPUTE_FORCE = False
PRECOMPUTE_FROM_RAW = False
PRECOMPUTE_DEPTH_RESIZE_MODE = "image"  # image/native/max_edge
PRECOMPUTE_DEPTH_MAX_EDGE = 480
PRECOMPUTE_SKIP_UNDISTORT = False
PRECOMPUTE_SKIP_CONFIDENCE = False
PRECOMPUTE_CV2_THREADS = None
PRECOMPUTE_WORKERS = None

print("Configuration loaded.")
print(f"  USE_KEYFRAMES: {USE_KEYFRAMES}")
print(f"  FRAME_STRIDE: {FRAME_STRIDE}")
print(f"  MAX_FRAMES: {MAX_FRAMES}")
print(f"  VOXEL_SIZE_M: {VOXEL_SIZE_M}m")
print(f"  Training iterations: {TRAINING_CONFIG['max_num_iterations']}")
print(f"  Cull alpha thresh: {TRAINING_CONFIG['cull_alpha_thresh']}")

# If using precomputed data, skip eager caching at startup
if USE_PRECOMPUTED_DATASET:
    CACHE_IMAGES_DEVICE = "lazy"


## 2) Download + extract bundle from Google Drive

In [ ]:
import importlib


def extract_gdrive_file_id(url: str) -> str | None:
    if not url:
        return None
    # Patterns: .../file/d/<ID>/view or id=<ID>
    match = re.search(r"/d/([a-zA-Z0-9_-]+)", url)
    if match:
        return match.group(1)
    match = re.search(r"id=([a-zA-Z0-9_-]+)", url)
    if match:
        return match.group(1)
    return None


def download_from_gdrive(url: str, out_path: Path) -> Path:
    if out_path.exists():
        return out_path
    file_id = extract_gdrive_file_id(url)
    if not file_id:
        raise ValueError("Could not extract file id from GDRIVE_URL. Use a share link or '?id=' URL.")

    gdown = importlib.import_module("gdown")
    download_url = f"https://drive.google.com/uc?id={file_id}"
    gdown.download(download_url, str(out_path), quiet=False)
    if not out_path.exists():
        raise RuntimeError("Download failed; file not found after gdown.")
    return out_path


zip_path = DOWNLOAD_DIR / "capture_bundle.zip"
if GDRIVE_URL:
    zip_path = download_from_gdrive(GDRIVE_URL, zip_path)
else:
    raise ValueError("Set GDRIVE_URL before running this cell.")

# Extract
bundle_root = EXTRACT_DIR / zip_path.stem
if not bundle_root.exists():
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(bundle_root)

# Detect precomputed dataset (contains transforms.json)
precomputed_candidates = [p.parent for p in bundle_root.rglob("transforms.json") if p.is_file()]
precomputed_candidates = [p for p in precomputed_candidates if "colmap" not in p.parts]
if precomputed_candidates:
    dataset_dir = precomputed_candidates[0]
    USING_PRECOMPUTED_DATASET_DOWNLOAD = True
    USE_PRECOMPUTED_DATASET = True
    PRECOMPUTE_FROM_RAW = False
    CACHE_IMAGES_DEVICE = "lazy"
    print("Found precomputed dataset:", dataset_dir)
else:
    USING_PRECOMPUTED_DATASET_DOWNLOAD = False
    # Find CaptureBundle_* folder
    bundle_dirs = [p for p in bundle_root.rglob("CaptureBundle_*") if p.is_dir()]
    non_macos = [p for p in bundle_dirs if "__MACOSX" not in p.parts]
    if not bundle_dirs:
        raise FileNotFoundError("No CaptureBundle_* folder found in extracted archive")
    bundle_dir = non_macos[0] if non_macos else bundle_dirs[0]
    print("Found bundle:", bundle_dir)


## 3) Inspect bundle + load anchors

In [ ]:
if USING_PRECOMPUTED_DATASET_DOWNLOAD:
    print("Skipping bundle inspection; using precomputed dataset.")
else:
    manifest = json.loads((bundle_dir / "manifest.json").read_text())
    anchors = json.loads((bundle_dir / "anchors.json").read_text())
    
    print("Bundle ID:", manifest.get("bundleId"))
    print("Frame count:", manifest.get("captureStats", {}).get("frameCount"))
    print("Keyframe count:", manifest.get("captureStats", {}).get("keyframeCount"))
    print("Depth enabled:", manifest.get("captureSettings", {}).get("depthEnabled"))
    print("Coordinate conventions:", manifest.get("coordinateConventions", {}))
    print("Capture duration:", f"{manifest.get('captureStats', {}).get('durationSeconds', 0):.1f}s")
    
    # Warn if using keyframes when we have many more frames
    frame_count = manifest.get("captureStats", {}).get("frameCount", 0)
    keyframe_count = manifest.get("captureStats", {}).get("keyframeCount", 0)
    if USE_KEYFRAMES and frame_count > keyframe_count * 2:
        print(f"\n⚠️  WARNING: Using only {keyframe_count} keyframes out of {frame_count} total frames.")
        print("    Consider setting USE_KEYFRAMES=False for better quality.")
    
    layout = manifest.get("coordinateConventions", {}).get("matrixLayout")
    if layout and layout != "row_major":
        raise ValueError(f"This bundle uses matrixLayout={layout}. Re-export with row_major to avoid transpose in training.")
    
    root_anchor = anchors.get("rootAnchor", {})
    root_transform = root_anchor.get("transform")
    if not root_transform:
        raise ValueError("anchors.json missing rootAnchor.transform")
    
    # Row-major 4x4
    T_root_world = np.array(root_transform, dtype=np.float32)
    print("\nRoot anchor transform (row-major):\n", T_root_world)

## 4) Select frames

By default, uses ALL captured frames (not just keyframes) for better quality.

In [ ]:
if USING_PRECOMPUTED_DATASET_DOWNLOAD:
    print("Skipping frame selection; using precomputed dataset.")
else:
    keyframes_dir = bundle_dir / "keyframes"
    frames_meta_dir = bundle_dir / "frames" / "meta"
    frames_rgb_dir = bundle_dir / "frames" / "rgb"
    frames_depth_dir = bundle_dir / "frames" / "depth"
    frames_confidence_dir = bundle_dir / "frames" / "confidence"
    
    if USE_KEYFRAMES:
        frame_ids = sorted([int(p.stem) for p in keyframes_dir.glob("*.jpg")])
        print(f"Using KEYFRAMES only")
    else:
        frame_ids = sorted([int(p.stem) for p in frames_rgb_dir.glob("*.jpg")])
        print(f"Using ALL frames")
    
    # Apply stride and max
    frame_ids = frame_ids[::FRAME_STRIDE]
    if MAX_FRAMES:
        frame_ids = frame_ids[:MAX_FRAMES]
    
    print(f"Selected {len(frame_ids)} frames (stride={FRAME_STRIDE}, max={MAX_FRAMES})")

## 5) Build Nerfstudio dataset (images + transforms.json)

In [ ]:
if USING_PRECOMPUTED_DATASET_DOWNLOAD:
    print("Skipping dataset build; using precomputed dataset.")
else:
    from datetime import datetime
    
    
    def load_frame_meta(frame_id: int) -> dict:
        meta_path = frames_meta_dir / f"{frame_id:06d}.json"
        if not meta_path.exists():
            raise FileNotFoundError(f"Missing meta for frame {frame_id}: {meta_path}")
        return json.loads(meta_path.read_text())
    
    
    def rows_to_matrix(rows: list[list[float]]) -> np.ndarray:
        return np.array(rows, dtype=np.float32)
    
    
    def intrinsics_from_meta(meta: dict):
        intr = meta["camera"]["intrinsics"]
        fx = intr[0][0]
        fy = intr[1][1]
        cx = intr[0][2]
        cy = intr[1][2]
        w = meta["camera"]["imageResolution"]["width"]
        h = meta["camera"]["imageResolution"]["height"]
        return fx, fy, cx, cy, w, h
    
    
    # Dataset target
    dataset_name = bundle_dir.name
    dataset_dir = DATASET_DIR / dataset_name
    images_dir = dataset_dir / "images"
    depth_out_dir = dataset_dir / "depth"
    confidence_out_dir = dataset_dir / "confidence"
    images_dir.mkdir(parents=True, exist_ok=True)
    depth_out_dir.mkdir(parents=True, exist_ok=True)
    confidence_out_dir.mkdir(parents=True, exist_ok=True)
    
    frames = []
    intrinsics_list = []
    
    for frame_id in tqdm(frame_ids, desc="Processing frames"):
        meta = load_frame_meta(frame_id)
        fx, fy, cx, cy, w, h = intrinsics_from_meta(meta)
        intrinsics_list.append([fx, fy, cx, cy, w, h])
    
        # Copy image - always from frames/rgb for consistency (even for keyframe IDs)
        src_img = frames_rgb_dir / f"{frame_id:06d}.jpg"
        if not src_img.exists() and USE_KEYFRAMES:
            # Fallback to keyframes dir if frame not in rgb
            src_img = keyframes_dir / f"{frame_id:06d}.jpg"
        dst_img = images_dir / f"{frame_id:06d}.jpg"
        if not dst_img.exists():
            shutil.copy2(src_img, dst_img)
    
        # Camera transform: row-major
        T_cam_world = rows_to_matrix(meta["camera"]["transform"])
        T_cam_anchor = np.linalg.inv(T_root_world) @ T_cam_world
        if APPLY_CAMERA_AXIS_CONVERSION:
            T_cam_anchor = T_cam_anchor @ CAMERA_AXIS_CONVERSION
    
        frame_entry = {
            "file_path": f"images/{frame_id:06d}.jpg",
            "transform_matrix": T_cam_anchor.tolist()
        }
    
        depth_src = frames_depth_dir / f"{frame_id:06d}.png"
        if depth_src.exists():
            depth_dst = depth_out_dir / f"{frame_id:06d}.png"
            if not depth_dst.exists():
                shutil.copy2(depth_src, depth_dst)
            frame_entry["depth_file_path"] = f"depth/{frame_id:06d}.png"
    
            if USE_CONFIDENCE:
                confidence_src = frames_confidence_dir / f"{frame_id:06d}.png"
                if confidence_src.exists():
                    confidence_dst = confidence_out_dir / f"{frame_id:06d}.png"
                    if not confidence_dst.exists():
                        shutil.copy2(confidence_src, confidence_dst)
                    frame_entry["confidence_file_path"] = f"confidence/{frame_id:06d}.png"
    
        frames.append(frame_entry)
    
    # Check intrinsics consistency
    intrinsics_arr = np.array(intrinsics_list)
    max_spread = intrinsics_arr.max(axis=0) - intrinsics_arr.min(axis=0)
    print("Intrinsics max spread [fx, fy, cx, cy, w, h]:", max_spread)
    
    use_per_frame_intrinsics = (
        USE_PER_FRAME_INTRINSICS
        or np.any(max_spread[:4] > INTRINSICS_SPREAD_THRESHOLD)
        or np.any(max_spread[4:] > 0)
    )
    print("Using per-frame intrinsics:", use_per_frame_intrinsics)
    
    if use_per_frame_intrinsics:
        for frame_entry, (fx, fy, cx, cy, w, h) in zip(frames, intrinsics_list):
            frame_entry.update({
                "fl_x": float(fx),
                "fl_y": float(fy),
                "cx": float(cx),
                "cy": float(cy),
                "w": int(w),
                "h": int(h)
            })
        transforms = {
            "camera_model": "OPENCV",
            "ply_file_path": "sparse_pc.ply",
            "frames": frames
        }
    else:
        fx, fy, cx, cy, w, h = intrinsics_list[0]
        transforms = {
            "camera_model": "OPENCV",
            "fl_x": float(fx),
            "fl_y": float(fy),
            "cx": float(cx),
            "cy": float(cy),
            "w": int(w),
            "h": int(h),
            "ply_file_path": "sparse_pc.ply",
            "frames": frames
        }
    
    (dataset_dir / "transforms.json").write_text(json.dumps(transforms, indent=2))
    print(f"Wrote transforms.json to {dataset_dir}")
    print(f"  - {len(frames)} frames")
    if use_per_frame_intrinsics:
        print("  - Intrinsics stored per-frame")
    else:
        print(f"  - Image resolution: {w}x{h}")
        print(f"  - Focal length: fx={fx:.1f}, fy={fy:.1f}")


In [ ]:
import sys
import subprocess
from pathlib import Path

if USE_PRECOMPUTED_DATASET and PRECOMPUTE_FROM_RAW:
    precomputed_dir = PRECOMPUTED_DATASET_DIR
    script_path = Path.cwd() / "precompute_dataset.py"
    if not script_path.exists():
        raise FileNotFoundError(f"Missing precompute script at {script_path}")
    if PRECOMPUTE_FORCE or not precomputed_dir.exists():
        cmd = [
            sys.executable,
            str(script_path),
            "--data", str(dataset_dir),
            "--output", str(precomputed_dir),
            "--depth-resize-mode", PRECOMPUTE_DEPTH_RESIZE_MODE,
            "--depth-max-edge", str(PRECOMPUTE_DEPTH_MAX_EDGE),
        ]
        if PRECOMPUTE_FORCE:
            cmd.append("--overwrite")
        if PRECOMPUTE_SKIP_UNDISTORT:
            cmd.append("--skip-undistort")
        if PRECOMPUTE_SKIP_CONFIDENCE:
            cmd.append("--skip-confidence")
        if PRECOMPUTE_CV2_THREADS is not None:
            cmd.extend(["--cv2-threads", str(PRECOMPUTE_CV2_THREADS)])
        if PRECOMPUTE_WORKERS is not None:
            cmd.extend(["--workers", str(PRECOMPUTE_WORKERS)])
        print("Running precompute:", " ".join(cmd))
        subprocess.run(cmd, check=True)
    dataset_dir = precomputed_dir
    print(f"Using precomputed dataset: {dataset_dir}")


## 6) Build `sparse_pc.ply` from depth

This gives Splatfacto a strong geometric initialization. Better init = better results.

In [ ]:
if USING_PRECOMPUTED_DATASET_DOWNLOAD:
    print("Skipping sparse point cloud build; using precomputed dataset.")
else:
    def load_depth_m(path: Path) -> np.ndarray:
        depth = imageio.imread(path)
        if depth.dtype != np.uint16:
            depth = depth.astype(np.uint16)
        return depth.astype(np.float32) / 1000.0  # mm -> meters
    
    
    def load_confidence(path: Path) -> np.ndarray:
        conf = imageio.imread(path)
        if conf.dtype != np.uint8:
            conf = conf.astype(np.uint8)
        return conf
    
    
    def unproject_depth(
        depth_m: np.ndarray,
        fx: float,
        fy: float,
        cx: float,
        cy: float,
        stride: int,
        confidence: np.ndarray | None = None,
        min_confidence: int | None = None
    ):
        h, w = depth_m.shape
        us = np.arange(0, w, stride)
        vs = np.arange(0, h, stride)
        uu, vv = np.meshgrid(us, vs)
        z = depth_m[np.ix_(vs, us)]
        mask = (z > MIN_DEPTH_M) & (z < MAX_DEPTH_M)
    
        if confidence is not None and min_confidence is not None:
            conf_h, conf_w = confidence.shape
            u_conf = np.clip((uu * (conf_w / w)).astype(np.int32), 0, conf_w - 1)
            v_conf = np.clip((vv * (conf_h / h)).astype(np.int32), 0, conf_h - 1)
            conf_vals = confidence[v_conf, u_conf]
            mask &= conf_vals >= min_confidence
    
        u = uu[mask]
        v = vv[mask]
        z = z[mask]
    
        x = (u - cx) / fx * z
        y = (v - cy) / fy * z
        pts = np.stack([x, y, z], axis=1).astype(np.float32)
        return pts, (u, v)
    
    
    def voxel_downsample(points: np.ndarray, colors: np.ndarray | None, voxel_size: float):
        if voxel_size <= 0:
            return points, colors
        voxel = np.floor(points / voxel_size).astype(np.int32)
        _, unique_idx = np.unique(voxel, axis=0, return_index=True)
        if colors is None:
            return points[unique_idx], None
        return points[unique_idx], colors[unique_idx]
    
    
    def write_ply_ascii(path: Path, points: np.ndarray, colors: np.ndarray | None = None):
        with path.open("w", encoding="ascii") as f:
            f.write("ply\nformat ascii 1.0\n")
            f.write(f"element vertex {len(points)}\n")
            f.write("property float x\nproperty float y\nproperty float z\n")
            if colors is not None:
                f.write("property uchar red\nproperty uchar green\nproperty uchar blue\n")
            f.write("end_header\n")
            if colors is None:
                for x, y, z in points:
                    f.write(f"{x} {y} {z}\n")
            else:
                for (x, y, z), (r, g, b) in zip(points, colors):
                    f.write(f"{x} {y} {z} {int(r)} {int(g)} {int(b)}\n")
    
    
    all_points = []
    all_colors = []
    
    # Use ALL frames for depth (not just selected training frames) for better init
    depth_frame_ids = sorted([int(p.stem) for p in frames_depth_dir.glob("*.png")])
    print(f"Building sparse point cloud from {len(depth_frame_ids)} depth frames...")
    
    for frame_id in tqdm(depth_frame_ids, desc="Depth to points"):
        meta = load_frame_meta(frame_id)
        depth_meta = meta.get("depth") or {}
        if not depth_meta.get("available"):
            continue
    
        depth_path = frames_depth_dir / f"{frame_id:06d}.png"
        if not depth_path.exists():
            continue
    
        depth_m = load_depth_m(depth_path)
        confidence = None
        if USE_CONFIDENCE:
            conf_path = frames_confidence_dir / f"{frame_id:06d}.png"
            if conf_path.exists():
                confidence = load_confidence(conf_path)
        # Scale intrinsics to depth resolution
        fx, fy, cx, cy, w, h = intrinsics_from_meta(meta)
        depth_h, depth_w = depth_m.shape
        fx_d = fx * (depth_w / w)
        fy_d = fy * (depth_h / h)
        cx_d = cx * (depth_w / w)
        cy_d = cy * (depth_h / h)
    
        pts_cam, (u_depth, v_depth) = unproject_depth(
            depth_m,
            fx_d,
            fy_d,
            cx_d,
            cy_d,
            DEPTH_STRIDE,
            confidence=confidence,
            min_confidence=MIN_CONFIDENCE if confidence is not None else None
        )
        if pts_cam.shape[0] == 0:
            continue
    
        # Transform to anchor space
        T_cam_world = rows_to_matrix(meta["camera"]["transform"])
        T_cam_anchor = np.linalg.inv(T_root_world) @ T_cam_world
        if APPLY_CAMERA_AXIS_CONVERSION:
            T_cam_anchor = T_cam_anchor @ CAMERA_AXIS_CONVERSION
    
        ones = np.ones((pts_cam.shape[0], 1), dtype=np.float32)
        pts_h = np.concatenate([pts_cam, ones], axis=1)
        pts_anchor = (T_cam_anchor @ pts_h.T).T[:, :3]
    
        # Sample colors from RGB
        img_path = frames_rgb_dir / f"{frame_id:06d}.jpg"
        if not img_path.exists():
            img_path = keyframes_dir / f"{frame_id:06d}.jpg"
        if img_path.exists():
            rgb = np.array(Image.open(img_path).convert("RGB"))
            rgb_h, rgb_w, _ = rgb.shape
            u_rgb = np.clip((u_depth * (rgb_w / depth_w)).astype(np.int32), 0, rgb_w - 1)
            v_rgb = np.clip((v_depth * (rgb_h / depth_h)).astype(np.int32), 0, rgb_h - 1)
            colors = rgb[v_rgb, u_rgb]
        else:
            colors = np.full((pts_anchor.shape[0], 3), 128, dtype=np.uint8)
    
        if MAX_POINTS_PER_FRAME and pts_anchor.shape[0] > MAX_POINTS_PER_FRAME:
            idx = np.random.choice(pts_anchor.shape[0], MAX_POINTS_PER_FRAME, replace=False)
            pts_anchor = pts_anchor[idx]
            colors = colors[idx]
    
        all_points.append(pts_anchor)
        all_colors.append(colors)
    
    if not all_points:
        raise RuntimeError("No depth points generated. Check depth availability and paths.")
    
    points = np.concatenate(all_points, axis=0)
    colors = np.concatenate(all_colors, axis=0)
    print(f"Raw points: {points.shape[0]:,}")
    
    points, colors = voxel_downsample(points, colors, VOXEL_SIZE_M)
    print(f"After voxel downsampling ({VOXEL_SIZE_M*100:.0f}cm): {points.shape[0]:,}")
    
    sparse_path = dataset_dir / "sparse_pc.ply"
    write_ply_ascii(sparse_path, points, colors)
    print(f"Wrote sparse point cloud: {sparse_path}")

## 7) Sanity checks

In [ ]:
# Basic bbox sanity check
mins = points.min(axis=0)
maxs = points.max(axis=0)
extents = maxs - mins
print("Point cloud bounds (m):")
print(f"  min: [{mins[0]:.2f}, {mins[1]:.2f}, {mins[2]:.2f}]")
print(f"  max: [{maxs[0]:.2f}, {maxs[1]:.2f}, {maxs[2]:.2f}]")
print(f"  extents: [{extents[0]:.2f}, {extents[1]:.2f}, {extents[2]:.2f}]m")

# Warn if scene is very large
if extents.max() > 20:
    print("\n⚠️  WARNING: Scene extents are large. Check for outlier depth values.")

# Quick 2D scatter preview (top-down XZ)
import matplotlib.pyplot as plt

sample_count = min(10000, points.shape[0])
sample = points[np.random.choice(points.shape[0], sample_count, replace=False)]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Top-down view (X-Z)
axes[0].scatter(sample[:, 0], sample[:, 2], s=1, alpha=0.5)
axes[0].set_title("Top-down view (X-Z)")
axes[0].set_xlabel("X (m)")
axes[0].set_ylabel("Z (m)")
axes[0].axis("equal")

# Side view (X-Y)
axes[1].scatter(sample[:, 0], sample[:, 1], s=1, alpha=0.5)
axes[1].set_title("Side view (X-Y)")
axes[1].set_xlabel("X (m)")
axes[1].set_ylabel("Y (m)")
axes[1].axis("equal")

plt.tight_layout()
plt.show()

## 7.5) Preflight checks

Make sure the dataset and sparse point cloud are ready before training.

In [ ]:
import json
from pathlib import Path
import numpy as np

transforms_path = dataset_dir / "transforms.json"
if not transforms_path.exists():
    raise FileNotFoundError(f"Missing transforms.json at {transforms_path}")

transforms = json.loads(transforms_path.read_text())
frames = transforms.get("frames", [])
if not frames:
    raise ValueError("transforms.json has no frames")

ply_rel = transforms.get("ply_file_path")
if not ply_rel:
    raise ValueError("transforms.json missing ply_file_path for sparse init")
ply_path = dataset_dir / ply_rel
if not ply_path.exists():
    raise FileNotFoundError(f"Sparse point cloud not found: {ply_path}")

# Spot-check image files
missing = []
for frame in frames[:10]:
    img = dataset_dir / frame["file_path"]
    if not img.exists():
        missing.append(str(img))
if missing:
    raise FileNotFoundError(f"Missing image files: {missing}")

# Spot-check transform matrices
for frame in frames[:5]:
    mat = np.array(frame.get("transform_matrix"))
    if mat.shape != (4, 4):
        raise ValueError(f"transform_matrix wrong shape: {mat.shape}")
    if not np.isfinite(mat).all():
        raise ValueError("transform_matrix has NaNs/Infs")

ply_size_mb = ply_path.stat().st_size / (1024 * 1024)
print(f"✅ Preflight OK:")
print(f"   - {len(frames)} training frames")
print(f"   - sparse_pc.ply present ({ply_size_mb:.1f} MB)")
print(f"   - Sample images verified")
print(f"   - Transform matrices valid")
print(f"\n   Dataset path: {dataset_dir}")

## 8) Train Splatfacto (Nerfstudio)

Run training on the GPU VM. This command will create an output folder under `outputs/`.

**Key hyperparameters for quality:**
- `cull_alpha_thresh`: Lower = keep more semi-transparent gaussians (default 0.1, we use 0.005)
- `densify_grad_thresh`: Lower = more aggressive densification (default 0.0008)
- `continue_cull_post_densification`: False = don't cull after 15k steps
- `use_scale_regularization`: True = reduce spikey artifacts

In [ ]:
import subprocess
import re
import sys
import time
from datetime import datetime

# Build training command from config
# Note: Nerfstudio 1.1.5 removed continue_cull_post_densification parameter
cmd_parts = [
    "ns-train", "splatfacto",
    "--vis", "tensorboard",
    "--logging.steps-per-log", str(LOGGING_STEPS_PER_LOG),
    "--logging.local-writer.enable", "False",  # Disable rich console output
    f"--max-num-iterations", str(TRAINING_CONFIG["max_num_iterations"]),
    f"--pipeline.model.cull-alpha-thresh", str(TRAINING_CONFIG["cull_alpha_thresh"]),
    f"--pipeline.model.densify-grad-thresh", str(TRAINING_CONFIG["densify_grad_thresh"]),
    f"--pipeline.model.num-downscales", str(TRAINING_CONFIG["num_downscales"]),
    f"--pipeline.model.resolution-schedule", str(TRAINING_CONFIG["resolution_schedule"]),
    f"--pipeline.model.refine-every", str(TRAINING_CONFIG["refine_every"]),
    f"--pipeline.model.stop-split-at", str(TRAINING_CONFIG["stop_split_at"]),
    f"--pipeline.model.ssim-lambda", str(TRAINING_CONFIG["ssim_lambda"]),
    f"--pipeline.model.use-tf32", str(USE_TF32),
    f"--pipeline.model.camera-optimizer-device", CAMERA_OPTIMIZER_DEVICE,
    f"--pipeline.model.max-gauss-ratio", str(TRAINING_CONFIG["max_gauss_ratio"]),
]

# Scale regularization to reduce spikey artifacts
if TRAINING_CONFIG["use_scale_regularization"]:
    cmd_parts.append("--pipeline.model.use-scale-regularization")
    cmd_parts.append("True")

# Pose refinement (camera optimizer)
if ENABLE_CAMERA_OPTIMIZER:
    cmd_parts.extend([
        "--pipeline.model.camera-optimizer.mode", CAMERA_OPTIMIZER_MODE
    ])

# Depth supervision (requires patched splatfacto + depth/ confidence in dataset)
if ENABLE_DEPTH_SUPERVISION:
    cmd_parts.extend([
        "--pipeline.model.output-depth-during-training", "True",
        "--pipeline.model.depth-loss-weight", str(DEPTH_LOSS_WEIGHT),
        "--pipeline.model.depth-loss-type", DEPTH_LOSS_TYPE,
        "--pipeline.model.depth-loss-huber-delta", str(DEPTH_LOSS_HUBER_DELTA),
        "--pipeline.model.depth-loss-ramp-steps", str(DEPTH_LOSS_RAMP_STEPS),
        "--pipeline.model.depth-min", str(DEPTH_SUP_MIN_M),
        "--pipeline.model.depth-max", str(DEPTH_SUP_MAX_M),
        "--pipeline.model.depth-confidence-min", str(DEPTH_SUP_MIN_CONFIDENCE),
    ])

# Datamanager performance/memory tweaks
cmd_parts.extend([
    "--pipeline.datamanager.cache-images", CACHE_IMAGES_DEVICE,
    "--pipeline.datamanager.cache-images-type", CACHE_IMAGES_TYPE,
    "--pipeline.datamanager.allow-large-gpu-cache", str(ALLOW_LARGE_GPU_CACHE),
    "--pipeline.datamanager.depth-resize-mode", DEPTH_RESIZE_MODE,
    "--pipeline.datamanager.depth-max-edge", str(DEPTH_MAX_EDGE),
    "--pipeline.datamanager.depth-dtype", DEPTH_DTYPE,
    "--pipeline.datamanager.confidence-dtype", CONFIDENCE_DTYPE,
    "--pipeline.datamanager.preprocessed-data", str(USE_PRECOMPUTED_DATASET),
    "--pipeline.datamanager.gpu-cache-max-fraction", str(GPU_CACHE_MAX_FRACTION),
    "--pipeline.datamanager.prefetch-to-gpu", str(PREFETCH_TO_GPU),
    "--pipeline.datamanager.non-blocking-transfers", str(NON_BLOCKING_TRANSFERS),
])
if MAX_THREAD_WORKERS is not None:
    cmd_parts.extend(["--pipeline.datamanager.max-thread-workers", str(MAX_THREAD_WORKERS)])
if CV2_NUM_THREADS is not None:
    cmd_parts.extend(["--pipeline.datamanager.cv2-num-threads", str(CV2_NUM_THREADS)])

# Dataparser args
cmd_parts.extend([
    "nerfstudio-data",
    "--data", str(dataset_dir),
    "--load-3D-points", "True",
])

print("Training command:")
print(" ".join(cmd_parts))
print()

In [ ]:
# Execute training with a live progress bar
import subprocess
import re
import sys
import os
from datetime import datetime
from tqdm import tqdm

# Patterns to extract step info from nerfstudio output
step_pattern = re.compile(r"[Ss]tep[:\s]+(\d+)")
loss_pattern = re.compile(r"loss[=:\s]+([\d.]+)")
psnr_pattern = re.compile(r"psnr[=:\s]+([\d.]+)", re.IGNORECASE)
gaussians_pattern = re.compile(r"num[_\s]gaussians[=:\s]+([\d,]+)", re.IGNORECASE)

print(f"Starting training at {datetime.now().strftime('%H:%M:%S')}")
print(f"Max iterations: {TRAINING_CONFIG['max_num_iterations']}")
print(f"Logging every {LOGGING_STEPS_PER_LOG} steps")
print("=" * 60)

start_time = datetime.now()

# Set environment to disable rich/fancy output
env = os.environ.copy()
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env["TERM"] = "dumb"
env["NO_COLOR"] = "1"
env["FORCE_COLOR"] = "0"

process = subprocess.Popen(
    cmd_parts,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env
)

progress = tqdm(
    total=TRAINING_CONFIG["max_num_iterations"],
    desc="Training",
    unit="step",
    smoothing=0.1
)

last_step = 0
current_loss = None
current_psnr = None
current_gaussians = None
buffer = ""

try:
    for line in process.stdout:
        buffer += line
        line = line.strip()
        if not line:
            continue

        step_match = step_pattern.search(line)
        if step_match:
            step = int(step_match.group(1))
            if step > last_step:
                progress.update(step - last_step)
                last_step = step

        loss_match = loss_pattern.search(line)
        if loss_match:
            current_loss = float(loss_match.group(1))

        psnr_match = psnr_pattern.search(line)
        if psnr_match:
            current_psnr = float(psnr_match.group(1))

        gauss_match = gaussians_pattern.search(line)
        if gauss_match:
            current_gaussians = gauss_match.group(1).replace(",", "")

        if step_match:
            postfix = {}
            if current_loss is not None:
                postfix["loss"] = f"{current_loss:.4f}"
            if current_psnr is not None:
                postfix["psnr"] = f"{current_psnr:.2f}"
            if current_gaussians is not None:
                postfix["gaussians"] = current_gaussians
            if postfix:
                progress.set_postfix(postfix)

        if any(kw in line.lower() for kw in [
            "error", "exception", "failed", "saving", "checkpoint", "finished", "complete",
            "caching / undistorting", "loading", "initializing"
        ]):
            tqdm.write(line)

except KeyboardInterrupt:
    tqdm.write("Training interrupted by user.")
    process.terminate()
finally:
    progress.close()

process.wait()
elapsed = (datetime.now() - start_time).total_seconds()

print("=" * 60)
if process.returncode == 0:
    print(f"✅ Training completed in {elapsed/60:.1f} minutes")
else:
    print(f"❌ Training failed with return code {process.returncode}")
    print("Last output:")
    print(buffer[-2000:] if len(buffer) > 2000 else buffer)


## 9) Export Gaussian Splat PLY

After training completes, export the PLY.

In [ ]:
# Auto-detect latest Nerfstudio run and export PLY
outputs_root = Path("outputs")
config_paths = list(outputs_root.rglob("config.yml"))
if not config_paths:
    raise FileNotFoundError("No config.yml found under outputs/. Run training first.")

latest_config = max(config_paths, key=lambda p: p.stat().st_mtime)
CONFIG_PATH = latest_config
print("Using config:", CONFIG_PATH)

OUT_DIR = EXPORT_DIR / dataset_dir.name
OUT_DIR.mkdir(parents=True, exist_ok=True)

!ns-export gaussian-splat --load-config "{CONFIG_PATH}" --output-dir "{OUT_DIR}"

## 9.5) Locate exported PLY

Lists exported PLY files and sizes so you can download the right one.

In [ ]:
from pathlib import Path

ply_files = sorted(OUT_DIR.glob("*.ply"))
if not ply_files:
    raise FileNotFoundError(f"No .ply files found in {OUT_DIR}")

print(f"Export directory: {OUT_DIR}")
print("\nExported files:")
for p in ply_files:
    size_mb = p.stat().st_size / (1024 * 1024)
    print(f"  - {p.name} ({size_mb:.2f} MB)")

# Estimate splat count from file size (rough: ~250 bytes per splat)
main_ply = ply_files[0]
estimated_splats = int(main_ply.stat().st_size / 250)
print(f"\nEstimated splat count: ~{estimated_splats:,}")
print(f"\niOS rendering expectations:")
print(f"  Quality mode: All splats at ~15fps")
print(f"  Balanced mode: Up to 60K splats")
print(f"  Fast mode: Up to 20K splats")

## 10) Alignment metadata (model → capture anchor)

Nerfstudio applies an orientation/centering transform and a scale during training. We invert those to align the exported PLY back into your capture anchor space.

In [ ]:
# Alignment from dataparser_transforms.json (no YAML parsing needed)
outputs_root = Path("outputs")
transform_paths = list(outputs_root.rglob("dataparser_transforms.json"))
if not transform_paths:
    raise FileNotFoundError("No dataparser_transforms.json found under outputs/. Run training first.")

latest_transform = max(transform_paths, key=lambda p: p.stat().st_mtime)
RUN_DIR = latest_transform.parent
print("Using run:", RUN_DIR)

# Where the exported PLY lives
OUT_DIR = EXPORT_DIR / dataset_dir.name
OUT_DIR.mkdir(parents=True, exist_ok=True)

payload = json.loads(latest_transform.read_text())
transform = payload.get("transform")
scale = float(payload.get("scale", 1.0))

if transform is None:
    raise ValueError("dataparser_transforms.json missing 'transform'")

T = np.array(transform, dtype=np.float32)
if T.shape == (3, 4):
    T4 = np.eye(4, dtype=np.float32)
    T4[:3, :4] = T
elif T.shape == (4, 4):
    T4 = T
else:
    raise ValueError(f"Unexpected transform shape: {T.shape}")

inv_T = np.linalg.inv(T4)
S_inv = np.diag([1.0 / scale, 1.0 / scale, 1.0 / scale, 1.0]).astype(np.float32)

# model (dataparser space) -> capture anchor space
model_to_anchor = inv_T @ S_inv

alignment = {
    "model_to_anchor_4x4": model_to_anchor.tolist(),
    "matrixLayout": "row_major"
}

alignment_path = OUT_DIR / "alignment.json"
alignment_path.write_text(json.dumps(alignment, indent=2))
print(f"Wrote {alignment_path}")
print(f"\nScale factor: {scale}")
print(f"Alignment matrix:\n{model_to_anchor}")

## Summary

Training complete! Your exported files are in the `exports/` directory.

**To use in DreamBrush iOS app:**
1. Download the `.ply` file and `alignment.json`
2. Create a SplatPackage with both files
3. Import into the app's Library

**If quality is still not satisfactory, try:**
- Capturing with better coverage (more angles, slower movement)
- Using a smaller, simpler scene
- Increasing `max_num_iterations` to 50000
- Decreasing `cull_alpha_thresh` further (e.g., 0.001)
- Using a newer iPhone with better LiDAR

**References:**
- [Splatfacto documentation](https://docs.nerf.studio/nerfology/methods/splat.html)
- [Nerfstudio hyperparameters](https://github.com/nerfstudio-project/nerfstudio/blob/main/nerfstudio/models/splatfacto.py)